# Multi-Scholar Islamic Fatwa RAG — Run on Google Colab
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbderrahimZineddineEsi/BinBaz-RAG/blob/multi-sheikh-rag/notebooks/02_run_on_colab.ipynb)

Comprehensive semantic search, neural reranking, and grounded relevance verification over **40,000+ fatwas** across major Sunni / Salafi scholars:
- **Sheikh Ibn Baz** (~19,700 fatwas)
- **Sheikh Ibn Uthaymeen** (~9,100 fatwas)
- **Al-Lajnah Ad-Da'imah** (Council of Senior Scholars, ~9,100 fatwas)
- **Sheikh Saleh Al-Fawzan** (~1,700 fatwas)
- **Sheikh Abdulaziz Aal ash-Sheikh, Abdullah al-Manee, Saleh bin Humaid**

---
### Key Features:
- **🎯 Fine-Tuned Laya Judge (Free & Offline)**: Powered by [`ZenoAbderrahim/laya-multilingual-fatwa-relevance`](https://huggingface.co/ZenoAbderrahim/laya-multilingual-fatwa-relevance) (322M mmBERT). Evaluates fatwa relevance directly on GPU (~30ms) with **86.4% recall** and **83.5% accuracy** without needing any API key!
- **🌐 Multi-Provider Flexibility (OpenRouter & Gemini)**: Full support to switch anytime to Google Gemini (`gemini-2.5-flash`, `gemini-2.5-pro`) or OpenRouter (470+ models including Claude 3.5 Sonnet, GPT-4o-mini, DeepSeek R1, JEV) for rewriting, judging, consensus classification, and comparative summaries.
- **☀️/🌙 Light & Dark Mode**: Seamless toggle switch in the top bar.
- **🔑 Zero Keys Required Out-of-the-Box**: Search, rerank, and Laya judge run entirely on Colab hardware. API keys in Settings/Secrets are only needed for generative LLM summaries.

---
### How to run:
1. **Runtime** → **Change runtime type** → **T4 GPU**.
2. (Optional) Set **Colab Secrets** (key icon on left sidebar):
   - `GEMINI_API_KEY`: Enables consensus classification & comparative summary.
   - `OPENROUTER_API_KEY`: Enables OpenRouter models and JEV verification.
3. Run the cells in order.


In [ ]:
# Step 1: Verify GPU
import torch
assert torch.cuda.is_available(), "No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU"
print("✓ GPU available:", torch.cuda.get_device_name(0))
print(f"  Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")

In [ ]:
# Step 2: Clone repository branch and install dependencies
!git clone -q -b multi-sheikh-rag https://github.com/AbderrahimZineddineEsi/BinBaz-RAG.git
%cd BinBaz-RAG
!pip install -q -r requirements.txt
print("✓ Repository and packages installed successfully.")

### Step 3: Multi-Scholar Dataset & Embeddings
The multi-scholar corpus requires three files inside `data/` (~226 MB total):
- `multi_scholar_clean.csv`
- `qwen3_06b_multi_scholar_embeddings.pt`
- `multi_scholar_metadata.json`

Choose **any ONE** of the following options to provide the files:

In [ ]:
# OPTION A: If you uploaded the 3 files to Hugging Face dataset (ZenoAbderrahim/binbaz_fatwas_rag)
# app.py will download them automatically when launching. Nothing to do here!

# OPTION B: Copy from Google Drive (Uncomment if you saved them in your Google Drive)
# from google.colab import drive
# import os, shutil
# drive.mount('/content/drive')
# os.makedirs('data', exist_ok=True)
# !cp /content/drive/MyDrive/kaggle_embedding_outputs/* data/

# OPTION C: Direct upload via Colab sidebar
# Open the file sidebar on the left -> BinBaz-RAG -> data -> Drag & drop the 3 files from your computer.

In [ ]:
# Verify dataset status in data/
import os
from pathlib import Path
import pandas as pd

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
multi_csv = data_dir / "multi_scholar_clean.csv"
multi_emb = data_dir / "qwen3_06b_multi_scholar_embeddings.pt"

if multi_csv.exists() and multi_emb.exists():
    df = pd.read_csv(multi_csv)
    print(f"✓ Multi-scholar dataset READY: {len(df):,} fatwas across {df['source'].nunique()} scholars:")
    for scholar, count in df['source'].value_counts().items():
        print(f"  • {scholar:25s}: {count:,} fatwas")
else:
    print("ℹ Multi-scholar files not found locally in data/.")
    print("  When running app.py in Step 5, it will attempt to download them from Hugging Face.")
    print("  (Or copy them to data/ using Option B or C above).")

In [ ]:
# Step 4: Load API Keys from Colab Secrets (Optional)
import os
try:
    from google.colab import userdata
    for name in ("GEMINI_API_KEY", "OPENROUTER_API_KEY"):
        try:
            val = userdata.get(name)
            if val:
                os.environ[name] = val
                print(f"✓ {name} loaded from Colab Secrets")
        except Exception:
            pass
except ImportError:
    pass

if not os.environ.get("GEMINI_API_KEY") and not os.environ.get("OPENROUTER_API_KEY"):
    print("ℹ No API keys provided — Running in 100% Free & Local mode!")
    print("  • Semantic Search (Qwen 0.6B) : READY on GPU")
    print("  • Mizan Reranker V2           : READY on GPU")
    print("  • Fine-Tuned Laya Judge       : READY on GPU (86.4% recall, 0 cost)")
    print("  (Add GEMINI_API_KEY to Colab Secrets if you also want query translation & consensus summaries).")


In [ ]:
# Step 5: Launch Gradio Web App with public share URL
# Embedder, reranker, and fine-tuned Laya judge all run on the Colab T4 GPU
!python app.py --share --no-browser --embedder-device cuda --reranker-device cuda --judge-device cuda
